In [1]:

from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def getPdfDocs(pdfDir):
    allDocs = []
    pobj = Path(pdfDir)
    if not pobj.is_dir():
        print(f"Dir Not Found")
        return None
    print(pobj.rglob("**/*.pdf"))
    pdf_files = pobj.rglob("**/*.pdf")
    # procces the pdf files 
    for pdf_file in pdf_files:
        print(f"Processing :{pdf_file}")
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        print(f"Loaded {len(docs)} pages")
        for doc in docs:
            doc.metadata["file_name"] = pdf_file.stem
            doc.metadata["file_type"] = pdf_file.suffix
        allDocs.extend(docs)
    return allDocs
all_docs = getPdfDocs("../doc_files/")
all_docs

C:\Users\Vivek\AppData\Local\Temp\ipykernel_3240\1873411810.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\Vivek\Desktop\agentic_ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<generator object Path.rglob at 0x0000026BF5CAF890>
Processing :..\doc_files\SowmyaG_Resume.pdf
Loaded 1 pages
Processing :..\doc_files\SPL.pdf
Loaded 6 pages
Processing :..\doc_files\VivekanandNaikCV.pdf
Loaded 1 pages
Processing :..\doc_files\Vivekanand_Naik_Resume.pdf
Loaded 1 pages


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-08-03T16:14:20+05:30', 'author': 'Un-named', 'moddate': '2026-08-03T16:14:20+05:30', 'source': '..\\doc_files\\SowmyaG_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': 'SowmyaG_Resume', 'file_type': '.pdf'}, page_content='SOWMYA G \nSoftware Developer | Python & Django Developer | Backend Development \n+91-8147588578 | g34402284@gmail.com | github.com/SowmyaGopal12 | linkedin//Sowmya G \nSUMMARY \nComputer Science undergraduate skilled in Python, Java, and Django, with hands -on experience building full-stack style applications \nusing object-oriented design and clean UI principles. Experienced building CRUD -driven applications with structured data \nmanagement, exception handling, and input validation. Strong foundation in Object -Oriented Programming, file handling, and version \ncontrol through Git and GitHub. \nTECHNICAL SKILLS \nLanguages: Pyth

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


# a text splitter function
def split_doc(docs, chunkSize=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunkSize,
        chunk_overlap = chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
        length_function =len
    )
    splitted_doc = text_splitter.split_documents(docs)
    print(f"splitted {len(docs)} Docs into {len(splitted_doc)}")
    # print(f"Content: {splitted_doc[0].page_content[:200]}")
    return splitted_doc
splitted_docs = split_doc(all_docs)
splitted_docs

splitted 9 Docs into 23


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-08-03T16:14:20+05:30', 'author': 'Un-named', 'moddate': '2026-08-03T16:14:20+05:30', 'source': '..\\doc_files\\SowmyaG_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': 'SowmyaG_Resume', 'file_type': '.pdf'}, page_content='SOWMYA G \nSoftware Developer | Python & Django Developer | Backend Development \n+91-8147588578 | g34402284@gmail.com | github.com/SowmyaGopal12 | linkedin//Sowmya G \nSUMMARY \nComputer Science undergraduate skilled in Python, Java, and Django, with hands -on experience building full-stack style applications \nusing object-oriented design and clean UI principles. Experienced building CRUD -driven applications with structured data \nmanagement, exception handling, and input validation. Strong foundation in Object -Oriented Programming, file handling, and version \ncontrol through Git and GitHub. \nTECHNICAL SKILLS \nLanguages: Pyth

In [3]:
from sentence_transformers import SentenceTransformer
from typing_extensions import List, Any

class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = None
        self.model_name = model_name
        self._load_model()
    
    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Dimension : {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model: {self.model_name} : {e}")
            raise ValueError("Model Cannot Be Loaded...")
        
    def generate_embedding(self, texts: List[str]):
        try:
            encodings = self.model.encode(texts, show_progress_bar=True)
            return encodings
        except Exception as e :
            print(e)
            
    def get_embedding_dimension(self):
        if not self.model:
            raise ValueError("Model Doesn't Exist")
        return self.model.get_embedding_dimension()

embedding_manager = EmbeddingManager()
embedding_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5257.75it/s]


Model Dimension : 384


In [ ]:
# import chromadb
# import os
# import uuid
# from typing_extensions import List, Any
# import numpy as np

# # Vector Store (chromaDB)
# class VectorStore:
#     def __init__(self, collection_name: str = "PDF_Collection", persist_dir: str = "./vector_store"):
#         self.collection_name = collection_name
#         self.persist_dir = persist_dir
#         self.collection = None
#         self.client = None
#         self._load_vectorDB()
    
#     # load the colleciton and client
#     def _load_vectorDB(self):
#         # initialize collection and client
#         os.makedirs(self.persist_dir, exist_ok=True)
#         try:
#             self.client = chromadb.PersistentClient(path=self.persist_dir)
#             # self.client.delete_collection(name=self.collection_name)
#             self.collection = self.client.get_or_create_collection(
#                 self.collection_name, 
#                 configuration={
#                   "hnsw": {
#                       "space": "cosine"
#                   }
#                 },
#                 metadata={
#                     "description": "PDF files for RAG"
#                     }
#                 )
#             print(f"Vector Store initialized successfully {self.collection_name}")
#             print(f"Existing Documents in collection {self.collection.count()}")
#         except Exception as e:
#             print(e)    
#     # Add docs to vector store 
#     def add_docs(self, docs: List[Any], embeddings: np.ndarray):
#         # prepare data 
#         doc_ids = []
#         doc_contents = []
#         doc_embeddings = []
#         metadatas = []
#         for i, (doc, embedding) in enumerate(zip(docs, embeddings)):
#             # unique id for each doc
#             uid = f"{uuid.uuid4().hex[:8]}_{i}"
#             doc_ids.append(uid)
            
#             # prepare page content
#             doc_contents.append(doc.page_content)
            
#             # embeddings 
#             doc_embeddings.append(embedding.tolist())
            
#             # Prepare metadata
#             metadata = dict(doc.metadata)
#             metadata["document_index"] = i
#             metadata["content_length"] = len(doc.page_content)
#             metadatas.append(metadata)
#             print(f"metadata: {metadata}")
            
#         # add fields to vector Store 
#         try:
#             self.collection.add(
#                 ids=doc_ids,
#                 documents=doc_contents,
#                 embeddings=doc_embeddings,
#                 metadatas=metadatas
#             )
#             print(f"Successfully added {len(docs)} into vector store...")
#         except Exception as e :
#             print(e)


# vectorstore_manager = VectorStore()

In [4]:
import typesense
import os
import uuid
from typing_extensions import List, Any
import numpy as np

KEY = os.environ["TYPESENSE_API_KEY"]

# Vector Store (typesense)
class TypesenseStore:
    def __init__(self, collection_name: str = "PDF_Collection", persist_dir: str = "./vector_store"):
        self.collection_name = collection_name
        self.persist_dir = persist_dir
        self.collection = None
        self.client = None
        self._load_vectorDB()
    
    # load the colleciton and client
    def _load_vectorDB(self):
        # initialize collection and client
        os.makedirs(self.persist_dir, exist_ok=True)
        try:
            self.client = typesense.Client({
                'nodes': [{
                'host': '765u8sqxmt9hiblcp-1.a1.typesense.net', # For Typesense Cloud use xxx.a1.typesense.net
                'port': '443',      # For Typesense Cloud use 443
                'protocol': 'https'   # For Typesense Cloud use https
                }],
                'api_key': f'{KEY}',
                'connection_timeout_seconds': 120
            })
            # self.client.delete_collection(name=self.collection_name)
            self.collection = self.client.collections.create({
                'name':self.collection_name,
                "fields": [
                    {
                        "name": "content",
                        "type": "string"
                    },
                    {
                        "name": "embedding",
                        "type": "float[]",
                        "num_dim": 384
                    },
                    {
                        "name": "document_index",
                        "type": "int32"
                    },
                    {
                        "name": "content_length",
                        "type": "int32"
                    },
                    {
                        "name": "metadata",
                        "type": "object"
                    }
                ],
                'enable_nested_fields':True
            })
            print(f"Vector Store initialized successfully {self.collection_name}")
            print(f"Existing Documents in collection {len(self.collection.keys())}")
        except Exception as e:
            print(e)    
    # Add docs to vector store 
    def add_docs(self, docs: List[Any], embeddings: np.ndarray):
        # prepare data 
        documents = []
        for i, (doc, embedding) in enumerate(zip(docs, embeddings)):
            # unique id for each doc
            uid = f"{uuid.uuid4().hex[:8]}_{i}"
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["document_index"] = i
            metadata["content_length"] = len(doc.page_content)
            
            document = {
                "id": uid,
                "content": doc.page_content,
                "embedding": embedding.tolist(),
                "document_index": i,
                "content_length": len(doc.page_content),
                "metadata": metadata
            }
            documents.append(document)

            
        # add fields to vector Store 
        try:
            self.client.collections[self.collection_name].documents.import_(
                documents,
                {"action": "create"}
            )            
            print(f"Successfully added {len(docs)} into vector store...")
        except Exception as e :
            print(e)

    # def search(self, query: str, query_by: str = "", *args):
    #     return self.client.collections['books'].documents.search(search_parameters)
typesense_manager = TypesenseStore()

Vector Store initialized successfully PDF_Collection
Existing Documents in collection 10


In [5]:
texts = [doc.page_content for doc in splitted_docs]
chunks_embeddings = embedding_manager.generate_embedding(texts)

typesense_manager.add_docs(splitted_docs, chunks_embeddings)

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.39it/s]


Successfully added 23 into vector store...


In [6]:
from typing_extensions import Dict

# Reterival Pipeline
class ReterivalManager:
    def __init__(self, embedding_manager: EmbeddingManager, vector_store: TypesenseStore):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store
    
    def reteriveContext(self, query: str, top_k: int, threshold: float = 0.0) -> list[Dict[str, Any]]:
        # Generate the qurey embedding 
        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        query_vector = query_embedding.tolist()
        
        # serach in vector store
        try:
            search_requests = {
                "searches": [
                    {
                        "collection": self.vector_store.collection_name,
                        "q": query,
                        "query_by": "content",
                        "vector_query": f"embedding:({query_vector}, k:{top_k})",
                        "exclude_fields": "embedding"
                    }
                ]
            }

            results = self.vector_store.client.multi_search.perform(
                search_requests,
                {}
            )
            search_result = results["results"][0]
            retrieved_docs = []

            for rank, hit in enumerate(search_result["hits"], start=1):

                document = hit["document"]

                distance = hit.get("vector_distance")

                retrieved_docs.append({
                    "id": document["id"],
                    "content": document["content"],
                    "metadata": document["metadata"],
                    "distance": distance,
                    "rank": rank
                })

            print(f"Retrieved {len(retrieved_docs)} documents")

            return retrieved_docs
        except Exception as e:
            print(e)

reterival = ReterivalManager(embedding_manager, typesense_manager)
reterival

In [7]:
reterival.reteriveContext("Acknowledgment", 10)

Batches: 100%|██████████| 1/1 [00:00<00:00, 97.57it/s]


Retrieved 10 documents


[{'id': 'db93202c_9',
  'content': 'Acknowledgment  \nThe successful organization of Srinivas Premier League – Season 8 was made \npossible through the constant support and encouragement of Dr. Ramakrishna N \nHegde, Dean, SUIET. \nWe extend our sincere gratitude to: \n• Dr. Shifana Begum, HoD – CSE \n• Dr. K Rama Narasimha, HoD – Mechanical Engineering \n• Sports Coordinators Prof. Raghavendra Baliga and Prof. Mahesh Kumar V B \n• Faculty members, organizing committee members, volunteers, and student \ncoordinators \nTheir dedication and teamwork ensured the smooth and successful conduct of the \nevent. \nWe also thank all the players and supporters for their active participation and \nenthusiasm. \n  \n  \nConclusion  \nSrinivas Premier League (SPL) – Season 8 successfully fulfilled its objective of \npromoting sports culture, teamwork, and leadership among students. \nThe tournament not only provided a platform for showcasing cricketing talent but \nalso strengthened unity and camar

In [ ]:
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from typing import Any, List, Tuple, Dict
import re
load_dotenv()

class AdvanceRagImplementation:
    # init
    def __init__(self, agent: Any, reteriver: ReterivalManager):
        self.agent = agent
        self.reteriver = reteriver
        self.history = []
        
    # reterival
    def _reterive_doc(self, query: str, top_k: int, threshold: float)->list[Dict[str, Any]]:
        return self.reteriver.reteriveContext(query=query, top_k=top_k, threshold=threshold)
    
    # context with citations and source 
    def _build_context(self, reterived_doc: Any) -> Tuple[str, List[Dict]]:
        context = "".join(
            f"[{i}]\n{doc['content']}"
            for i, doc in enumerate(reterived_doc, start=1))
        
        sources = [
            {
                "citation_id": i,
                "chunk_id": doc.get("id", f"chunk_{i}"),
                "source": doc.get("metadata", {}).get(
                    "source",
                    doc.get("metadata", {}).get("source_file", "unknown"),
                ),
                "page": doc.get("metadata", {}).get("page", "unknown"),
                "score": doc.get("distance"),
                "preview": doc["content"][:120] + "...",
            }
            for i, doc in enumerate(reterived_doc, start=1)
        ]
        return context, sources
    
    # Build Input (sys msg, history , question/query)
    def _build_messages(
        self, query: str, context: str
    ) -> Tuple[SystemMessage, HumanMessage]:
        system_content = f"""You are a RAG assistant.
Use the conversation history only to understand the user's current question.
Use the retrieved context as the ONLY source of factual information.

Rules:
- Do not use outside knowledge.
- If the answer is not present in the retrieved context, say:
  "I don't have enough information in the provided documents."
- Cite factual statements using ONLY square brackets matching the context (e.g., [1], [2]).
- NEVER use any other citation format such as 【1】, (1), or [1†L1-L4].
- Put citations immediately after the supported statement.
- Do not invent citations.

Retrieved Context:
{context}"""

        return SystemMessage(content=system_content), HumanMessage(
            content=query
        )

    # Extract the cittaions from the answer 
    def _extract_citations(self, answer: str) -> List[int]:
        matches = re.findall(r"\[(\d+)\]|【(\d+)】", answer)
        return sorted({int(a or b) for a, b in matches})
    
    def _resolve_sources(
        self, answer: str, sources: List[Dict]
    ) -> List[Dict]:
        citation_map = {source["citation_id"]: source for source in sources}
        used_citations = self._extract_citations(answer)

        return [
            citation_map[citation_id]
            for citation_id in used_citations
            if citation_id in citation_map
        ]
        
    def query_fun(self, query: str, top_k: int = 5, threshold: float = 0.0, stream: bool = False, summarize: bool = False)-> tuple[Dict, Dict, Dict]:
        # 1. reteriver
        reterived_doc = self._reterive_doc(
            query=query, top_k=top_k, threshold=threshold
        )

        if not reterived_doc:
            print("\n[INFO] No Context Found")
            answer = "I don't have enough information in the provided documents."

            self.history.append(HumanMessage(content=query))
            self.history.append(AIMessage(content=answer))

            return {
                "question": query,
                "answer": answer,
                "sources": [],
            }

        
        # 2. build context and sources from reterived_doc
        context, sources = self._build_context(reterived_doc)
        
        print("\n========== CONTEXT ==========")
        print(context)

        print("\n========== SOURCES ==========")
        print(sources)
        # 3. set a fallback technique in case for no reterived doc 
        if not context:
            print("No Context found")
            answer = ""
            sources = []
            
        
        # 4. Buid the Message(System Message and Human Message)
        sys_message, human_message = self._build_messages(context, query)
        
        # 5. Build the execution payload (combine history + SystemMessage + query)
        input_message = [sys_message] + self.history + [human_message]
        
        answer = ""
        
        # 6. Model Generation 
        if stream:
            for chunk in self.agent.stream(
                {"messages": input_message},
                stream_mode="updates"
            ):
                if not chunk:
                    continue

                # chunk looks like:
                # {"model": {"messages": [AIMessage(...)]}}

                for node_output in chunk.values():

                    if not isinstance(node_output, dict):
                        continue

                    messages = node_output.get("messages", [])

                    for message in messages:

                        if isinstance(message, AIMessage):
                            content = message.content

                            if content:
                                answer += content
                                print(content, end="", flush=True)


        else:
            resp = self.agent.invoke({"messages": input_message})
            if isinstance(resp, dict) and "messages" in resp:
                answer = resp["messages"][-1].content

            elif hasattr(resp, "content"):
                answer = resp.content

            else:
                answer = str(resp)
        
        # 7. resolve used sources from the citations 
        used_sources = self._resolve_sources(answer, sources)
        
        # 8. update the history 
        self.history.append(sys_message)
        self.history.append(human_message)
        
        # 7. return eturn {'question', 'answer', 'sources', 'history'}
        return {
            "question": query,
            "answer": answer,
            "source": used_sources,
            "history": self.history
        }

# llm

llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0.0, max_tokens=1024, timeout=60, max_retries=2)
llm
summarizer = SummarizationMiddleware(
    model=llm,
    trigger=("tokens", 10000),
    keep=("messages", 4)
)
agent = create_agent(
    model=llm,
    tools=[],
    middleware=[summarizer]
)     
rag = AdvanceRagImplementation(agent=agent, reteriver=reterival)    


In [ ]:
# res = rag.query_fun("who is better Sowmya or vivekanand in terms of coding and development", stream=True)
res = rag.query_fun("So Who might lie in their resume", stream=True)


In [ ]:
res